In [1]:
# Import necessary libraries
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
from sklearn.neighbors import NearestNeighbors
import sys

# Debug: Confirm that imports are successful
print("Debug: Libraries imported successfully.")

# Print the versions of Python and each imported module
print(f"Python version: {sys.version}")
print(f"os version: Part of Python standard library, version {sys.version}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"geopandas version: {gpd.__version__}")
print(f"scikit-learn version: {NearestNeighbors.__module__.split('.')[1]}")  # Version info for sklearn

# Set the base directory for datasets in Kaggle
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets
sub_dir = r"/kaggle/working/"  # Submission directory for output files

Debug: Libraries imported successfully.
Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
os version: Part of Python standard library, version 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
numpy version: 1.26.4
pandas version: 2.2.3
geopandas version: 0.14.4
scikit-learn version: neighbors


In [2]:
# Debug: Print the directory paths to confirm they are set correctly
print(f"Debug: Base directory: {base_dir}")
print(f"Debug: Submission directory: {sub_dir}")

# Debug: List the files in the base directory to verify the presence of input files
print(f"Debug: Listing files in {base_dir}:")
try:
    files_in_dir = os.listdir(base_dir)
    print(files_in_dir)
except Exception as e:
    print(f"Error: Could not list files in {base_dir}. Error: {str(e)}")
    raise Exception(f"Failed to access directory {base_dir}")

# File paths for input datasets
MESONET_FILE = os.path.join(base_dir, "NY_Mesonet_Weather.xlsx")
TRAIN_FILE = os.path.join(base_dir, "Training_data.csv")
VALIDATION_FILE = os.path.join(base_dir, "Validation_data.csv")

# Intermediate output for weather data
WEATHER_OUTPUT = os.path.join(sub_dir, "mesonet_weather.csv")

# Final output CSVs for training and validation with weather data
TRAIN_OUTPUT = os.path.join(sub_dir, "training_data_with_weather.csv")
VALIDATION_OUTPUT = os.path.join(sub_dir, "validation_data_with_weather.csv")

# Debug: Check if input files exist
print(f"Debug: Mesonet weather file exists: {os.path.exists(MESONET_FILE)}")
print(f"Debug: Training dataset file exists: {os.path.exists(TRAIN_FILE)}")
print(f"Debug: Validation dataset file exists: {os.path.exists(VALIDATION_FILE)}")

Debug: Base directory: /kaggle/input/eyds-base-dataset
Debug: Submission directory: /kaggle/working/
Debug: Listing files in /kaggle/input/eyds-base-dataset:
['census_block_loc.csv', 'Hyperlocal_Temperature_Monitoring_20250311.csv', 'Airquality_Unique_geocode_with_LatLong.xlsx', 'nyclion_25a', 'StreetAssessmentRating', 'USA_wind-speed_10m.tif', 'USA_power-density_10m.tif', 'Validation_data.csv', 'LSAT_8_221022', 'Training_data.csv', 'NYC_Cooling_Tower_Registrations_20250224.csv', 'AQ', 'Automated_Traffic_Volume_Counts_20250319.csv', 'Air_Quality_20250221.csv', 'USA_air-density_10m.tif', 'nclimgrid-monthly-202107.tif', 'Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data_for_Calendar_Year_2021__20250224.csv', 'Building_Footprint.kml', 'Sentinel_3', 'Landsat_LST.ipynb', 'Landsat_NDVI.tiff', '2015_Street_Tree_Census_-_Tree_Data_20250221.csv', 'NY_Mesonet_Weather.xlsx', 'Landsat_LST.tiff', 'Building Footprints_20250222.geojson', 'nyc_census_tracts.csv']
Debug: Mesonet weather file

In [3]:
#############################################
# Part A: Process Mesonet Weather Data
#############################################

# 1. Read Mesonet sheets for Bronx and Manhattan
print("Loading Mesonet weather data...")
try:
    bronx_weather = pd.read_excel(MESONET_FILE, sheet_name="Bronx")
    manhattan_weather = pd.read_excel(MESONET_FILE, sheet_name="Manhattan")
except FileNotFoundError as e:
    print(f"Error: Mesonet weather file not found at {MESONET_FILE}. Error: {str(e)}")
    raise Exception("Failed to load Mesonet weather dataset")

print("Bronx weather data loaded with", bronx_weather.shape[0], "rows.")
print("Manhattan weather data loaded with", manhattan_weather.shape[0], "rows.")
print("Debug: Bronx weather columns:", bronx_weather.columns.tolist())
print("Debug: Manhattan weather columns:", manhattan_weather.columns.tolist())

# 2. Convert 'Date / Time' column to datetime
bronx_weather['Date / Time'] = pd.to_datetime(bronx_weather['Date / Time'])
manhattan_weather['Date / Time'] = pd.to_datetime(manhattan_weather['Date / Time'])

# 3. Define time window for filtering
time_filter_start = pd.Timestamp("2021-07-24 15:00:00")
time_filter_end = pd.Timestamp("2021-07-24 16:00:00")

# 4. Filter and compute average weather conditions
bronx_filtered = bronx_weather[(bronx_weather['Date / Time'] >= time_filter_start) &
                               (bronx_weather['Date / Time'] <= time_filter_end)]
manhattan_filtered = manhattan_weather[(manhattan_weather['Date / Time'] >= time_filter_start) &
                                       (manhattan_weather['Date / Time'] <= time_filter_end)]

# Debug: Check filtered data
print("Debug: Bronx filtered data rows:", bronx_filtered.shape[0])
print("Debug: Manhattan filtered data rows:", manhattan_filtered.shape[0])

# Compute averages
bronx_avg = bronx_filtered.mean(numeric_only=True)
manhattan_avg = manhattan_filtered.mean(numeric_only=True)

# 5. Create a DataFrame with weather features for each station, including coordinates
weather_df = pd.DataFrame({
    'station': ['Bronx', 'Manhattan'],
    'Latitude': [40.87248, 40.76754],  # Provided coordinates
    'Longitude': [-73.89352, -73.96449],
    'Altitude': [57.5, 94.8],
    'air_temp_surface': [bronx_avg['Air Temp at Surface [degC]'], manhattan_avg['Air Temp at Surface [degC]']],
    'relative_humidity': [bronx_avg['Relative Humidity [percent]'], manhattan_avg['Relative Humidity [percent]']],
    'wind_speed': [bronx_avg['Avg Wind Speed [m/s]'], manhattan_avg['Avg Wind Speed [m/s]']],
    'wind_direction': [bronx_avg['Wind Direction [degrees]'], manhattan_avg['Wind Direction [degrees]']],
    'solar_flux': [bronx_avg['Solar Flux [W/m^2]'], manhattan_avg['Solar Flux [W/m^2]']]
})

# Debug: Check the weather DataFrame
print("Debug: Weather DataFrame:")
print(weather_df)

# 6. Save the computed weather data to CSV for later use
weather_df.to_csv(WEATHER_OUTPUT, index=False)
print(f"Mesonet weather data saved to {WEATHER_OUTPUT}.")

# 7. Create a GeoDataFrame for the weather stations
weather_gdf = gpd.GeoDataFrame(
    weather_df,
    geometry=gpd.points_from_xy(weather_df['Longitude'], weather_df['Latitude']),
    crs="EPSG:4326"
)

Loading Mesonet weather data...
Bronx weather data loaded with 169 rows.
Manhattan weather data loaded with 169 rows.
Debug: Bronx weather columns: ['Date / Time', 'Air Temp at Surface [degC]', 'Relative Humidity [percent]', 'Avg Wind Speed [m/s]', 'Wind Direction [degrees]', 'Solar Flux [W/m^2]']
Debug: Manhattan weather columns: ['Date / Time', 'Air Temp at Surface [degC]', 'Relative Humidity [percent]', 'Avg Wind Speed [m/s]', 'Wind Direction [degrees]', 'Solar Flux [W/m^2]']
Debug: Bronx filtered data rows: 13
Debug: Manhattan filtered data rows: 13
Debug: Weather DataFrame:
     station  Latitude  Longitude  Altitude  air_temp_surface  \
0      Bronx  40.87248  -73.89352      57.5         27.492308   
1  Manhattan  40.76754  -73.96449      94.8         26.753846   

   relative_humidity  wind_speed  wind_direction  solar_flux  
0          44.615385    3.100000      135.076923  445.692308  
1          48.353846    2.946154      171.076923  441.000000  
Mesonet weather data saved to

<ipython-input-3-b1e2a0f58419>:20: FutureWarning: Parsed string "2021-07-24 06:00:00 EDT" included an un-recognized timezone "EDT". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  bronx_weather['Date / Time'] = pd.to_datetime(bronx_weather['Date / Time'])
<ipython-input-3-b1e2a0f58419>:21: FutureWarning: Parsed string "2021-07-24 06:00:00 EDT" included an un-recognized timezone "EDT". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  manhattan_weather['Date / Time'] = pd.to_datetime(manhattan_weather['Date / Time'])


In [4]:
#############################################
# Part B: Load Training and Validation Data
#############################################

# 8. Load the training and validation datasets and convert them to GeoDataFrames
print("Loading training data...")
try:
    train_df = pd.read_csv(TRAIN_FILE)
except FileNotFoundError:
    print(f"Error: Training data file not found at {TRAIN_FILE}")
    raise Exception("Failed to load training dataset")

print("Training data loaded with", train_df.shape[0], "rows.")
train_gdf = gpd.GeoDataFrame(
    train_df.copy().reset_index(),  # original index stored in "index"
    geometry=gpd.points_from_xy(train_df.Longitude, train_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Training GeoDataFrame shape: {train_gdf.shape}")
print(f"Debug: Training GeoDataFrame columns: {train_gdf.columns.tolist()}")

print("Loading validation data...")
try:
    val_df = pd.read_csv(VALIDATION_FILE)
except FileNotFoundError:
    print(f"Error: Validation data file not found at {VALIDATION_FILE}")
    raise Exception("Failed to load validation dataset")

print("Validation data loaded with", val_df.shape[0], "rows.")
val_gdf = gpd.GeoDataFrame(
    val_df.copy().reset_index(),
    geometry=gpd.points_from_xy(val_df.Longitude, val_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Validation GeoDataFrame shape: {val_gdf.shape}")
print(f"Debug: Validation GeoDataFrame columns: {val_gdf.columns.tolist()}")

Loading training data...
Training data loaded with 11229 rows.
Debug: Training GeoDataFrame shape: (11229, 6)
Debug: Training GeoDataFrame columns: ['index', 'Longitude', 'Latitude', 'datetime', 'UHI Index', 'geometry']
Loading validation data...
Validation data loaded with 1040 rows.
Debug: Validation GeoDataFrame shape: (1040, 5)
Debug: Validation GeoDataFrame columns: ['index', 'Longitude', 'Latitude', 'UHI Index', 'geometry']


In [5]:
#############################################
# Part C: Assign Weather Data Using KNN (Nearest Station)
#############################################

# 9. Use KNN to find the nearest weather station for each point in training and validation sets
# Extract coordinates of weather stations and training/validation points
weather_coords = weather_gdf[['Longitude', 'Latitude']].to_numpy()
train_coords = train_gdf[['Longitude', 'Latitude']].to_numpy()
val_coords = val_gdf[['Longitude', 'Latitude']].to_numpy()

# Fit KNN model to find the nearest weather station
knn = NearestNeighbors(n_neighbors=1, metric='euclidean')
knn.fit(weather_coords)

# Find the nearest weather station for training data
train_distances, train_indices = knn.kneighbors(train_coords)
train_gdf['nearest_station_idx'] = train_indices.flatten()
train_gdf['distance_to_station'] = train_distances.flatten()

# Find the nearest weather station for validation data
val_distances, val_indices = knn.kneighbors(val_coords)
val_gdf['nearest_station_idx'] = val_indices.flatten()
val_gdf['distance_to_station'] = val_distances.flatten()

# 10. Merge weather data with training and validation sets
# Map the weather data to the training and validation sets based on the nearest station
weather_features = ['station', 'air_temp_surface', 'relative_humidity', 'wind_speed', 'wind_direction', 'solar_flux']
train_joined = train_gdf.merge(
    weather_gdf[weather_features],
    left_on='nearest_station_idx',
    right_index=True,
    how='left',
    suffixes=('', '_weather')
)

val_joined = val_gdf.merge(
    weather_gdf[weather_features],
    left_on='nearest_station_idx',
    right_index=True,
    how='left',
    suffixes=('', '_weather')
)

# Debug: Check the joined data
print("Debug: Joined training data shape:", train_joined.shape)
print("Debug: Joined training data columns:", train_joined.columns.tolist())
print("Debug: Sample joined training data (first 5 rows):")
print(train_joined[['Longitude', 'Latitude', 'station', 'air_temp_surface', 'relative_humidity', 'wind_speed', 'wind_direction', 'solar_flux', 'distance_to_station']].head())

print("Debug: Joined validation data shape:", val_joined.shape)
print("Debug: Joined validation data columns:", val_joined.columns.tolist())
print("Debug: Sample joined validation data (first 5 rows):")
print(val_joined[['Longitude', 'Latitude', 'station', 'air_temp_surface', 'relative_humidity', 'wind_speed', 'wind_direction', 'solar_flux', 'distance_to_station']].head())

Debug: Joined training data shape: (11229, 14)
Debug: Joined training data columns: ['index', 'Longitude', 'Latitude', 'datetime', 'UHI Index', 'geometry', 'nearest_station_idx', 'distance_to_station', 'station', 'air_temp_surface', 'relative_humidity', 'wind_speed', 'wind_direction', 'solar_flux']
Debug: Sample joined training data (first 5 rows):
   Longitude   Latitude station  air_temp_surface  relative_humidity  \
0 -73.909167  40.813107   Bronx         27.492308          44.615385   
1 -73.909187  40.813045   Bronx         27.492308          44.615385   
2 -73.909215  40.812978   Bronx         27.492308          44.615385   
3 -73.909242  40.812908   Bronx         27.492308          44.615385   
4 -73.909257  40.812845   Bronx         27.492308          44.615385   

   wind_speed  wind_direction  solar_flux  distance_to_station  
0         3.1      135.076923  445.692308             0.061400  
1         3.1      135.076923  445.692308             0.061465  
2         3.1      13

In [6]:
#############################################
# Part D: Clean Up & Save Final CSVs
#############################################

# 11. Clean up the final DataFrames
cols_to_drop = ['geometry', 'nearest_station_idx', 'distance_to_station']
train_final = train_joined.drop(columns=cols_to_drop, errors='ignore')
val_final = val_joined.drop(columns=cols_to_drop, errors='ignore')

# 12. Save the final DataFrames to CSV
train_final.to_csv(TRAIN_OUTPUT, index=False)
val_final.to_csv(VALIDATION_OUTPUT, index=False)

# Debug: Print the final confirmation messages with file paths
print(f"Debug: Training data with weather features saved to: {TRAIN_OUTPUT}")
print(f"Debug: Validation data with weather features saved to: {VALIDATION_OUTPUT}")
print("Done! Weather columns added to both training and validation datasets.")
print(f"Train output: {TRAIN_OUTPUT}")
print(f"Validation output: {VALIDATION_OUTPUT}")

Debug: Training data with weather features saved to: /kaggle/working/training_data_with_weather.csv
Debug: Validation data with weather features saved to: /kaggle/working/validation_data_with_weather.csv
Done! Weather columns added to both training and validation datasets.
Train output: /kaggle/working/training_data_with_weather.csv
Validation output: /kaggle/working/validation_data_with_weather.csv
